In [65]:
# Load geojson as dataframe
import geopandas as gpd

investments_gdf = gpd.read_file("final/final_data_502.geojson")
income_gdf = gpd.read_file("final/final_data_prices_limits.geojson")

In [71]:
import json
from typing import Dict

import pandas as pd
import numpy as np

investment_col_descs = {
  "county_fips": {
    "description": "5-digit FIPS code for the county",
    "datatype": "string",
  },
  "state": {
    "description": "State name",
    "datatype": "string",
  },
  "county": {
    "description": "County name",
    "datatype": "string",
  },
  "year": {
    "description": "Year",
    "datatype": "integer",
  },
  "total_population": {
    "description": "Total population",
    "datatype": "integer",
  },
  "population_below_poverty": {
    "description": "Population below poverty",
    "datatype": "integer",
  },
  "population_below_poverty_percent": {
    "description": "Population below poverty percent",
    "datatype": "float",
  },
  "adults_25_and_older_less_than_high_school_graduate": {
    "description": "Adults 25 and older less than high school graduate",
    "datatype": "integer",
  },
  "adults_25_and_older_less_than_high_school_graduate_percent": {
    "description": "Adults 25 and older less than high school graduate percent",
    "datatype": "float",
  },
  "adults_25_and_older_with_bachelor's_degree_or_higher": {
    "description": "Adults 25 and older with bachelor's degree or higher",
    "datatype": "integer",
  },
  "adults_25_and_older_with_bachelor's_degree_or_higher_percent": {
    "description": "Adults 25 and older with bachelor's degree or higher percent",
    "datatype": "float",
  },
  "total_investment_dollars": {
    "description": "Total investment dollars",
    "datatype": "integer",
  },
  "total_number_of_investments": {
    "description": "Total number of investments",
    "datatype": "integer",
  },
  "number_of_households": {
    "description": "Number of households",
    "datatype": "integer",
  },
  "average_income_per_household": {
    "description": "Average income per household",
    "datatype": "float",
  },
  "total_median_earnings": {
    "description": "Total median earnings",
    "datatype": "float",
  },
  "geometry": {
    "description": "Geometry",
    "datatype": "geometry",
  },
}

income_col_descs = {
  "year": {
    "description": "Year",
    "datatype": "integer",
  },
  "county_fips": {
    "description": "5-digit FIPS code for the county",
    "datatype": "string",
  },
  "state": {
    "description": "State name",
    "datatype": "string",
  },
  "county": {
    "description": "County name",
    "datatype": "string",
  },
  "median_home_price": {
    "description": "Median home price",
    "datatype": "integer",
  },
  "income_limit_1_person": {
    "description": "Income limit for 1 person",
    "datatype": "integer",
  },
  "income_limit_2_person": {
    "description": "Income limit for 2 person",
    "datatype": "integer",
  },
  "income_limit_3_person": {
    "description": "Income limit for 3 person",
    "datatype": "integer",
  },
  "income_limit_4_person": {
    "description": "Income limit for 4 person",
    "datatype": "integer",
  },
  "income_limit_5_person": {  
    "description": "Income limit for 5 person",
    "datatype": "integer",
  },
  "income_limit_6_person": {
    "description": "Income limit for 6 person",
    "datatype": "integer",
  },
  "income_limit_7_person": {
    "description": "Income limit for 7 person",
    "datatype": "integer",
  },
  "income_limit_8_person": {
    "description": "Income limit for 8 person",
    "datatype": "integer",
  },
  "geometry": {
    "description": "Geometry",
    "datatype": "geometry",
  },
}
# Ensure each column in gdf is the correct datatype as specified, and drop unnecessary columns

def clean_and_cast_dataframe(df, col_descs):
    """
    Given a DataFrame and a dictionary of column descriptions,
    - Returns a new DataFrame with only the columns present in col_descs.
    - Casts columns to the correct pandas dtype according to col_descs["datatype"].
    - Does not modify the original input DataFrame.
    """
    # Build mapping from column name to intended datatype
    col_datatypes = {k: v["datatype"] for k, v in col_descs.items()}

    # Create new DataFrame with only columns listed in col_descs, in order
    filtered_cols = [col for col in df.columns if col in col_descs]
    new_df = df.loc[:, filtered_cols].copy()

    # Ensure correct types for each column
    for col, dtype in col_datatypes.items():
        if col not in new_df.columns:
            continue
        # Map datatype field to pandas dtype
        if dtype == "integer":
            new_df[col] = new_df[col].astype("Int64")
        elif dtype == "float":
            new_df[col] = new_df[col].astype("float64")
        elif dtype == "string":
            new_df[col] = new_df[col].astype("string")
        elif dtype == "boolean":
            new_df[col] = new_df[col].astype("boolean")
        # geometry is handled automatically by geopandas (if applicable)

    return new_df


def normalize_example(x):
    # Converts numpy scalars to float/int, leaves normal Python types unchanged;
    # for non-numerics, leave unchanged except for numpy objects (e.g. np.str_)
    # Special handling for geometry objects: leave as-is
    # Also, convert pd.Timestamp/numpy datetime to ISO string
    if isinstance(x, (np.generic,)):
        if np.issubdtype(type(x), np.floating):
            return float(x)
        elif np.issubdtype(type(x), np.integer):
            return int(x)
        elif np.issubdtype(type(x), np.bool_):
            return bool(x)
        elif np.issubdtype(type(x), np.str_):
            return str(x)
        elif np.issubdtype(type(x), np.datetime64):
            return pd.Timestamp(x).isoformat()
        else:
            return x
    elif isinstance(x, pd.Timestamp):
        return x.isoformat()
    else:
        return x


def create_data_dictionary(gdf: gpd.GeoDataFrame, col_descs: Dict[str, str]):
    data_dictionary = []
    # Only use columns that are present in col_descs
    for col in col_descs.keys():
        if col not in gdf.columns:
            continue  # Skip if column is not in the GeoDataFrame
        
        series = gdf[col]
        dtype = str(series.dtype)
        # Nullable
        nullable = series.isnull().any()
        num_unique_values = series.nunique(dropna=True)
        num_null_values = int(series.isnull().sum())
        desc = col_descs[col]["description"]
        human_name = col.replace("_", " ").title()

        print(col, dtype)

        # Determine datatype similar to ColumnDatatype
        col_dtype = series.dtype

        if pd.api.types.is_integer_dtype(col_dtype):
            datatype = "integer"
        elif pd.api.types.is_float_dtype(col_dtype):
            datatype = "float"
        elif pd.api.types.is_bool_dtype(col_dtype):
            datatype = "boolean"
        elif col_dtype.name == "geometry":
            # handle geometry differently
            col_info = {
                "id": col,
                "name": col,
                "description": desc,
                "datatype": "geometry",
                "nullable": bool(nullable),
                "humanReadableName": human_name,
                "numUniqueValues": int(num_unique_values) if num_unique_values is not None else None,
                "numNullValues": int(num_null_values) if num_null_values is not None else None,
            }

            data_dictionary.append(col_info)
            continue
        else:
            datatype = "string"

        # min/max/length/unit
        min_val = None
        max_val = None
        length_val = None
        unit_val = None
        example_values = None
        possible_values = None

        if datatype in ("integer", "float"):
            if not series.empty:
                min_val = float(series.min()) if series.notnull().any() else None
                max_val = float(series.max()) if series.notnull().any() else None
            # Optionally: units if can be inferred from description
            desc_lc = col_descs[col]["description"].lower()
            
            if "percent" in desc_lc:
                unit_val = "percent"
            elif "usd" in desc_lc or "dollars" in desc_lc or "income" in desc_lc or "earnings" in desc_lc:
                unit_val = "USD"

        elif datatype == "string":
            try:
                length_val = int(series.map(lambda x: len(x) if isinstance(x, str) else 0).max())
            except Exception:
                length_val = None
            # possibleValues: for a reasonable string col
            n_unique = series.nunique(dropna=True)
            if n_unique > 0 and n_unique <= 20:
                possible_values = sorted([normalize_example(v) for v in series.dropna().unique()])
        elif datatype == "boolean":
            possible_values = [True, False]
        # Geometry typically no extra stats

        # numUnique/Null
        num_unique_values = series.nunique(dropna=True)
        num_null_values = int(series.isnull().sum())
        # Sample example values (show up to 5, if exist), normalize types
        if not series.dropna().empty:
            # list(series.dropna().unique()[:5]) —> need to convert np types to normal float/int
            examples = list(series.dropna().unique()[:5])
            example_values = [normalize_example(v) for v in examples]
        else:
            example_values = None


        # Compose full dictionary
        col_info = {
            "id": col,
            "name": col,
            "description": col_descs[col]["description"],
            "datatype": datatype,
            "nullable": bool(nullable),
            "humanReadableName": human_name,
            "min": min_val,
            "max": max_val,
            "unit": unit_val,
            "length": length_val,
            "possibleValues": possible_values,
            "exampleValues": example_values,
            "numUniqueValues": int(num_unique_values) if num_unique_values is not None else None,
            "numNullValues": int(num_null_values) if num_null_values is not None else None,
        }

        data_dictionary.append(col_info)

    return data_dictionary


cleaned_investments_gdf = clean_and_cast_dataframe(investments_gdf, investment_col_descs)
cleaned_income_gdf = clean_and_cast_dataframe(income_gdf, income_col_descs)

investment_columns = create_data_dictionary(investments_gdf, investment_col_descs)
income_columns = create_data_dictionary(income_gdf, income_col_descs)

investment_data_dict = {
  "id": "investments",
  "kind": "geospatial",
  "location": "internal",
  "name": "USDA 502 Loan Investments",
  "description": "USDA 502 loan investment data by county with demographic and economic indicators",
  "columns": investment_columns
}

income_data_dict = {
  "id": "income",
  "kind": "geospatial",
  "location": "internal",
  "name": "502 Loan Income Limits",
  "description": "502 Loan income limits by county",
  "columns": income_columns
}

with open("workspace/data/investments/data-dictionary.json", "w") as f:
    json.dump(investment_data_dict, f, indent=2)

with open("workspace/data/income/data-dictionary.json", "w") as f:
    json.dump(income_data_dict, f, indent=2)

print(cleaned_investments_gdf.dtypes)
print(cleaned_investments_gdf.head())

cleaned_investments_gdf.to_parquet("workspace/data/investments/block-0.parquet")
cleaned_investments_gdf.to_parquet("workspace/data/investments/full.parquet")
cleaned_income_gdf.to_parquet("workspace/data/income/block-0.parquet")
cleaned_income_gdf.to_parquet("workspace/data/income/full.parquet")


county_fips object
state object
county object
year int32
total_population object
population_below_poverty object
population_below_poverty_percent object
adults_25_and_older_less_than_high_school_graduate object
adults_25_and_older_less_than_high_school_graduate_percent float64
adults_25_and_older_with_bachelor's_degree_or_higher object
adults_25_and_older_with_bachelor's_degree_or_higher_percent float64
total_investment_dollars float64
total_number_of_investments int32
number_of_households object
average_income_per_household object
total_median_earnings object
geometry geometry
year int32
county_fips object
state object
county object
median_home_price object
income_limit_1_person int32
income_limit_2_person int32
income_limit_3_person int32
income_limit_4_person int32
income_limit_5_person int32
income_limit_6_person int32
income_limit_7_person int32
income_limit_8_person int32
geometry geometry
county_fips                                                              Int64
state       

In [80]:
# Calculate quintiles to color by for 'total_investment_dollars' in cleaned_investments_gdf
# Calculate quintile breakpoints for total_investment_dollars
quintile_breaks = cleaned_investments_gdf["total_investment_dollars"].quantile([0.2, 0.4, 0.6, 0.8]).values

# Print the quintile breakpoints and a sample
print("Quintile breakpoints (total_investment_dollars):", quintile_breaks)

# Print the quintile breakpoints for median home price
quintile_breaks = cleaned_income_gdf["median_home_price"].quantile([0.2, 0.4, 0.6, 0.8]).values

# Print the quintile breakpoints and a sample
print("Quintile breakpoints (median_home_price):", quintile_breaks)

# Create 8 break points evenly spaced between 60% and 140% of income_limit_4_person
min_val = cleaned_income_gdf["income_limit_4_person"].min() * 0.6
max_val = cleaned_income_gdf["income_limit_4_person"].max() * 1.4
breaks = [min_val + (max_val - min_val) * i / 7 for i in range(8)]

print("8 breakpoints (60% to 140% of income_limit_4_person):", breaks)

Quintile breakpoints (total_investment_dollars): <FloatingArray>
[996049.6000000001, 2557266.8000000007, 4839349.4, 9294704.8]
Length: 4, dtype: Float64
Quintile breakpoints (median_home_price): <FloatingArray>
[134820.0, 166900.0, 200490.0, 253250.0]
Length: 4, dtype: Float64
8 breakpoints (60% to 140% of income_limit_4_person): [np.float64(32640.0), np.float64(48397.142857142855), np.float64(64154.28571428571), np.float64(79911.42857142858), np.float64(95668.57142857142), np.float64(111425.71428571429), np.float64(127182.85714285714), np.float64(142940.0)]


In [48]:
gdf.head()

# Figure out quintiles to color by
# get the investments per capita (total_investment_dollars / total_population)
gdf["investment_dollars_per_capita"] = gdf["total_investment_dollars"] / gdf["total_population"]

# get the quintile breakpoints
quintile_breaks = gdf["investment_dollars_per_capita"].quantile([0.2, 0.4, 0.6, 0.8]).values

# assign quintile labels (1-5) to each row
gdf["investment_dollars_per_capita_quintile"] = pd.cut(
    gdf["investment_dollars_per_capita"],
    bins=[-float("inf")] + list(quintile_breaks) + [float("inf")],
    labels=[1, 2, 3, 4, 5]
)

# print the quintile breakpoints and head
print("Quintile breakpoints:", quintile_breaks)
print(gdf[["investment_dollars_per_capita", "investment_dollars_per_capita_quintile"]].head())


Quintile breakpoints: <FloatingArray>
[45.307023754385476, 81.05456270904729, 121.6377597017761, 191.98828354475916]
Length: 4, dtype: Float64
   investment_dollars_per_capita investment_dollars_per_capita_quintile
0                      61.341118                                      2
1                     156.784231                                      4
2                     101.920794                                      3
3                     116.117078                                      3
4                     108.741193                                      3


In [83]:
# # Do kmeans clustering to get 5 clusters
# from sklearn.cluster import KMeans

# # Drop rows with NaN in 'investment_dollars_per_capita' before clustering
# gdf_nonan = cleaned_investments_gdf.dropna(subset=["investment_dollars_per_capita"]).copy()

# kmeans = KMeans(n_clusters=5, random_state=42)
# clusters = kmeans.fit_predict(gdf_nonan[["investment_dollars_per_capita"]])

# # print the unique cluster labels
# print("Cluster labels:", pd.unique(clusters))

# Predict 8 clusters between 60% and 140% of income_limit_4_person and print the actual cluster break values (dollars)
from sklearn.cluster import KMeans
import numpy as np

# Calculate the range between 60% and 140%
min_income = cleaned_income_gdf["income_limit_4_person"].min() * 0.6
max_income = cleaned_income_gdf["income_limit_4_person"].max() * 1.4

# Drop rows with NaN and clip within [min_income, max_income] for clustering
income_nonan = cleaned_income_gdf.dropna(subset=["income_limit_4_person"]).copy()
income_vals = income_nonan["income_limit_4_person"].clip(lower=min_income, upper=max_income).values.reshape(-1, 1)

# Fit KMeans to find 8 clusters
kmeans8 = KMeans(n_clusters=8, random_state=42, n_init=10)
income_clusters = kmeans8.fit_predict(income_vals)

# Get the cluster centers and sort them for breakpoints
cluster_centers = np.sort(kmeans8.cluster_centers_.flatten())

# Midpoints between adjacent cluster centers as break values
income_breaks = list(cluster_centers)
print("8 income break values (cluster centers, dollars, 60%-140% range):", [round(b, 2) for b in income_breaks])


8 income break values (cluster centers, dollars, 60%-140% range): [np.float64(56832.32), np.float64(60413.24), np.float64(63025.0), np.float64(67453.16), np.float64(71640.0), np.float64(77474.19), np.float64(84469.23), np.float64(92247.37)]


In [76]:
# Run tippacanoe to generate pmtiles from geojson
# May need to cast ints to floats
import subprocess
from typing import Literal

# Convert gdf to FlatGeobuf format (.fgb)
cleaned_investments_gdf.to_file("workspace/maps/fahe/layers/investments/layer.fgb", driver="FlatGeobuf")
cleaned_income_gdf.to_file("workspace/maps/fahe/layers/income/layer.fgb", driver="FlatGeobuf")


def create_pmtiles(layer_name, col_descs):
    def get_pmtile_type_mapping(datatype: str) -> Literal["string", "float", "int", "bool"]:
        if datatype == "integer":
            return "int"
        elif datatype == "float":
            return "float"

    type_mapping = {
        col: get_pmtile_type_mapping(col_descs[col]["datatype"])
        for col in col_descs
        if col_descs[col]["datatype"] in ["integer", "float"]
    }

    print(type_mapping)

    cmd = [
        "tippecanoe",
        "-o",
        f"workspace/maps/fahe/layers/{layer_name}/layer.pmtiles",
        "-zg",
        "--drop-densest-as-needed",
        "--extend-zooms-if-still-dropping",
        *(
            [f"--attribute-type={col}:{type_mapping[col]}" for col in type_mapping]
            if type_mapping
            else []
        ),
        "-l",
        "fgb",
        f"workspace/maps/fahe/layers/{layer_name}/layer.fgb"
    ]

    subprocess.run(cmd, check=True)

create_pmtiles("investments", investment_col_descs)
create_pmtiles("income", income_col_descs)

{'county_fips': 'int', 'year': 'int', 'total_population': 'int', 'population_below_poverty': 'int', 'population_below_poverty_percent': 'float', 'adults_25_and_older_less_than_high_school_graduate': 'int', 'adults_25_and_older_less_than_high_school_graduate_percent': 'float', "adults_25_and_older_with_bachelor's_degree_or_higher": 'int', "adults_25_and_older_with_bachelor's_degree_or_higher_percent": 'float', 'total_investment_dollars': 'int', 'total_number_of_investments': 'int', 'number_of_households': 'int', 'average_income_per_household': 'float', 'total_median_earnings': 'float'}


detected indexed FlatGeobuf: assigning feature IDs by sequence
3750 features, 2051601 bytes of geometry and attributes, 224126 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z1 for features typically 148511 feet (45267 meters) apart, and at least 62780 feet (19136 meters) apart
Choosing a maxzoom of -z5 for resolution of about 9024 feet (2750 meters) within features
  99.9%  5/8/12  
detected indexed FlatGeobuf: assigning feature IDs by sequence


{'year': 'int', 'county_fips': 'int', 'median_home_price': 'int', 'income_limit_1_person': 'int', 'income_limit_2_person': 'int', 'income_limit_3_person': 'int', 'income_limit_4_person': 'int', 'income_limit_5_person': 'int', 'income_limit_6_person': 'int', 'income_limit_7_person': 'int', 'income_limit_8_person': 'int'}


411 features, 212912 bytes of geometry and attributes, 22224 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z1 for features typically 149867 feet (45680 meters) apart, and at least 62625 feet (19089 meters) apart
Choosing a maxzoom of -z5 for resolution of about 9209 feet (2806 meters) within features
  99.9%  5/8/12  


In [78]:
# Copy files to the public folder
import shutil

shutil.copy("workspace/maps/fahe/layers/investments/layer.pmtiles", "502-investments-map/public/investments.pmtiles")
shutil.copy("workspace/maps/fahe/layers/income/layer.pmtiles", "502-investments-map/public/income.pmtiles")

investments_gdf.to_file("502-investments-map/public/502-investments.geojson", driver="GeoJSON")
income_gdf.to_file("502-investments-map/public/income.geojson", driver="GeoJSON")

In [53]:
# Copy the geojson to the public folder
import shutil

shutil.copy("final/final_output.geojson", "502-investments-map/public/502-investments.geojson")


FileNotFoundError: [Errno 2] No such file or directory: 'final/final_output.geojson'